In [2]:
import sys
sys.path.insert(0, '..')

caminho_poemas_portinari = '../Data/poemas.csv'
caminho_poemas_extra = '../Data/portuguese-poems.csv'
caminho_noticias = '../Data/Historico_de_materias.csv'

In [3]:
import pandas as pd

portinari = pd.read_csv(caminho_poemas_portinari)[['estrofes']].rename(columns={'estrofes': 'texto'})
noticias  = pd.read_csv(caminho_noticias)[['conteudo_noticia']].rename(columns={'conteudo_noticia': 'texto'}).sample(n=111, random_state=42)
poemas    = pd.read_csv(caminho_poemas_extra)[['Content']].rename(columns={'Content': 'texto'}).sample(n=111, random_state=42)

portinari['label'] = 1
noticias['label']  = 0
poemas['label']    = 0

df_final = pd.concat([portinari, noticias, poemas]).sample(frac=1, random_state=42).reset_index(drop=True)

In [4]:
#Apenas para testar rapido
#df_final = df_final.sample(n=60, random_state=42)

In [ ]:
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

from transformers import BertTokenizerFast, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from sklearn.model_selection import train_test_split
import torch

MODEL = 'neuralmind/bert-base-portuguese-cased'
tokenizer = BertTokenizerFast.from_pretrained(MODEL)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando device: {device}")

train, val = train_test_split(df_final, test_size=0.2, stratify=df_final['label'])

def tokenize(batch):
    return tokenizer(batch['texto'], truncation=True, padding='max_length', max_length=512)

ds_train = Dataset.from_pandas(train).map(tokenize, batched=True, batch_size=32)
ds_val   = Dataset.from_pandas(val).map(tokenize, batched=True, batch_size=32)

model = BertForSequenceClassification.from_pretrained(MODEL, num_labels=2)



args = TrainingArguments(
    output_dir='./bertimbau-portinari',
    num_train_epochs=10,  # Mais épocas = melhor aprendizado
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    evaluation_strategy='steps',  # Avaliar a cada N steps
    eval_steps=5,  # Avaliar a cada 5 steps
    save_strategy='steps',
    save_steps=5,
    load_best_model_at_end=True,
    fp16=True,
    logging_steps=2,  # Log mais frequente
    learning_rate=5e-5,  # Taxa de aprendizado aumentada
    warmup_steps=50,  # Aquecimento gradual
    weight_decay=0.01,  # Regularização
    dataloader_num_workers=4,
    gradient_accumulation_steps=2,  # Simula batch maior
)

trainer = Trainer(model=model, args=args, train_dataset=ds_train, eval_dataset=ds_val)
trainer.train()

Usando device: cuda


/home/al.mateus.torres/.local/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Map:   0%|          | 0/266 [00:00<?, ? examples/s]

Map:   0%|          | 0/67 [00:00<?, ? examples/s]

/home/al.mateus.torres/.local/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

  0%|          | 0/2 [00:00<?, ?it/s]

{'eval_loss': 0.03431291505694389, 'eval_runtime': 0.3882, 'eval_samples_per_second': 172.605, 'eval_steps_per_second': 5.152, 'epoch': 7.78}
{'loss': 0.0546, 'grad_norm': 1.6906652450561523, 'learning_rate': 3.6e-05, 'epoch': 8.0}
{'loss': 0.0411, 'grad_norm': 5.332955837249756, 'learning_rate': 3.8e-05, 'epoch': 8.44}
{'loss': 0.0284, 'grad_norm': 0.40398117899894714, 'learning_rate': 4e-05, 'epoch': 8.89}


  0%|          | 0/2 [00:00<?, ?it/s]

{'eval_loss': 0.01634586602449417, 'eval_runtime': 0.3838, 'eval_samples_per_second': 174.577, 'eval_steps_per_second': 5.211, 'epoch': 8.89}
{'train_runtime': 43.9471, 'train_samples_per_second': 60.527, 'train_steps_per_second': 0.91, 'train_loss': 0.38426208551973107, 'epoch': 8.89}


TrainOutput(global_step=40, training_loss=0.38426208551973107, metrics={'train_runtime': 43.9471, 'train_samples_per_second': 60.527, 'train_steps_per_second': 0.91, 'total_flos': 627256755978240.0, 'train_loss': 0.38426208551973107, 'epoch': 8.88888888888889})

In [19]:
# Salvar o modelo treinado
trainer.save_model('./bertimbau-portinari')
model.save_pretrained('./bertimbau-portinari')
tokenizer.save_pretrained('./bertimbau-portinari')

print("✅ Modelo salvo com sucesso!")

# Agora carregue para testar
from transformers import pipeline

clf = pipeline('text-classification', model='./bertimbau-portinari', tokenizer=tokenizer)

# Teste com poema
poema2 = """De tudo, ao meu amor serei atento
Antes, e com tal zelo, e sempre, e tanto
Que mesmo em face do maior encanto
Dele se encante mais meu pensamento.

Quero vivê-lo em cada vão momento
E em louvor hei de espalhar meu canto
E rir meu riso e derramar meu pranto
Ao seu pesar ou seu contentamento.

E assim, quando mais tarde me procure
Quem sabe a morte, angústia de quem vive
Quem sabe a solidão, fim de quem ama

Eu possa me dizer do amor (que tive):
Que não seja imortal, posto que é chama
Mas que seja infinito enquanto dure."""
resultado1 = clf(poema2)
print("Poema Portinari:", resultado1)

poema_test = """Os retirantes vêm vindo com trouxas e embrulhos vêm das terras secas e escuras; 
pedregulhos doloridos como fagulhas de carvão aceso corpos disformes, uns panos sujos, rasgados 
e sem cor, dependurados homens de enorme ventre bojudo mulheres com trouxas caídas para o lado 
pançudas, carregando ao colo um garoto choramingando, remelento mocinhas de peito duro e vestido 
roto velhas trôpegas marcadas pelo tempo olhos de catarata e pés informes aos velhos cegos agarradas 
pés inchados enormes levantando o pó da cor de suas vestes rasgadas no rumor monótono das alparcatas 
há uma pausa, cai no pó a mulher que carrega uma lata de água! Só há umas gotas — dá uma só não vai 
arribar. É melhor o marido e os filhos ficarem. Nós vamos andando temos muito que andar neste chão 
batido as secas vão a morte semeando."""

resultado0 = clf(poema_test)
print("Poema portinari real:", resultado0)

# Teste com notícia
noticia = """Pelo menos seis turistas estrangeiros foram assaltados ao saírem da Pedra do Sal, reduto boêmio na região central do Rio de Janeiro, na madrugada desta terça-feira (19). Um deles foi agredido.

Foram duas ocorrências: uma envolvendo quatro americanos, outra com uma dupla de russos. Ambos os casos foram registrados na Delegacia Especial de Apoio ao Turismo (Deat), no Leblon.

O roubo aos americanos
Quatro americanos deixaram a Pedra do Sal com o guia de turismo e dois jovens brasileiras em dois táxis.

Segundo a Polícia Militar, por volta das 3h, criminosos de moto abordaram um dos veículos no Elevado Paulo de Frontin, antes da entrada do Túnel Rebouças, na pista sentido Lagoa.

"""
resultado2 = clf(noticia)
print("Notícia:", resultado2)

✅ Modelo salvo com sucesso!
Poema Portinari: [{'label': 'LABEL_0', 'score': 0.978766143321991}]
Poema portinari real: [{'label': 'LABEL_1', 'score': 0.7864364981651306}]
Notícia: [{'label': 'LABEL_0', 'score': 0.99233078956604}]
